In [1]:
import numpy as np
import pandas as pd
from pandas import Series, DataFrame

# Predicting E-Commerce Conversion Behavior

This project analyzes user session data from an onlinle cosmetic store to identify behavioral signals associated with purchase conversion.

The goals are:

* Identify behavioral patterns that distinguish converting sessions
* Build a predictive model of conversion probability
* Evaluating how these predictions could support targeted marketing strategies

## Data Preparation

The dataset contains event-level logs describing user actions such as product views, cart additions, and purchases.

To analyze user behavior, events are aggregated into session-level observations. Each row in the final dataset represents a single browsing session.

In [ ]:
df = pd.read_csv("../data/2019-Oct.csv")


In [3]:
df.isna().sum()

event_time             0
event_type             0
product_id             0
category_id            0
category_code    1032453
brand             416286
price                  0
user_id                0
user_session          85
dtype: int64

In [4]:
df = df.dropna(subset = ["user_session"])

In [5]:
before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Dropped {before - after} duplicate rows")

Dropped 57084 duplicate rows


## Feature Engineering

Session-level features are created to capture user engagement and browsing behavior.
These include:

* interaction intensity (views, events)
* session duration
* price exposure
* time-of-day behavior

In [6]:
# Create binary indicators for each event type in df
df["is_view"] = (df["event_type"] == "view").astype(int)
df["is_remove_cart"] = (df["event_type"] == "remove_from_cart").astype(int)
df["is_add_cart"] = (df["event_type"] == "cart").astype(int)
df["is_purchase"] = (df["event_type"] == "purchase").astype(int)

# Revenue and price columns used later for sessions level integration
df["purchase_revenue"] = df["price"] * df["is_purchase"]
df["view_price"] = df["price"] * df["is_view"]

# Correctly convert event_time to datetime for time based calculations in sessions table
df["event_time"] = pd.to_datetime(df["event_time"], utc=True)

# price only when product is viewed
df["view_price"] = df["price"].where(df["is_view"] == 1)

In [7]:
# Aggregate event-level data into session-level observations
# Each row represents a single user session

# Session-level engagement and price exposure metrics
sessions = (
    df
    .groupby(["user_id", "user_session"])
    .agg(
        session_start = ("event_time","min"),
        session_end = ("event_time", "max"),
        num_views = ("is_view","sum"),
        num_remove_cart = ("is_remove_cart","sum"),
        num_add_cart = ("is_add_cart","sum"),
        num_purchase = ("is_purchase","sum"),

        session_revenue = ("purchase_revenue","sum"),
        avg_view_price = ("view_price","mean"),
        max_view_price = ("view_price", "max"),

        num_unique_brand = ("brand","nunique")
    )
    .reset_index()
)

# Core session behavior metrics used in conversion analysis
sessions["converted"] = (sessions["num_purchase"] > 0).astype(int)
sessions["session_duration"] = (sessions["session_end"] - sessions["session_start"]).dt.total_seconds()
sessions["total_events"] = (sessions["num_views"] + sessions["num_remove_cart"] + sessions["num_add_cart"] + sessions["num_purchase"])
sessions["has_cart"] = (sessions["num_add_cart"] > 0)

# Time based session features
sessions["hour"] = (sessions["session_start"].dt.hour)
sessions["weekday"] =(sessions["session_start"].dt.weekday)
sessions["is_weekend"] = (sessions["weekday"].isin([5,6]).astype(int))

# Log transforms to reduce skew in engagement metrics
sessions["log_views"] = np.log1p(sessions["num_views"])
sessions["log_duration"] = np.log1p(sessions["session_duration"])

# Cyclical encoding so hour 23 and hour 0 are treated as close in time
sessions["hour_sin"] = np.sin(2 * np.pi * sessions["hour"] / 24)
sessions["hour_cos"] = np.cos(2 * np.pi * sessions["hour"] / 24)

# Funnel progression metrics within the session
sessions["view_to_cart_rate"] = (sessions["num_add_cart"] / sessions["num_views"])
sessions["cart_to_purchase_rate"] = (sessions["num_purchase"] / sessions["num_add_cart"])

# Handle sessions with no product views (no price exposure)
sessions["no_price_exposure"] = sessions["avg_view_price"].isna().astype(int)
sessions["avg_view_price"] = sessions["avg_view_price"].fillna(0)


In [ ]:
sessions.to_csv("../data/sessions.csv", index = False)